# 00 — G0 pilot: CausalGym × lượng tử hoá trọng số (Pythia)

**Mục tiêu cổng G0** (chưa phải kết quả khoa học):
1. Đo *t* = thời gian một lần học DAS 1D → ước lượng GPU-giờ cho MVP.
2. Odds của DAS trên FP ở vài (layer, vùng) → so bậc độ lớn với CausalGym.
3. Hành vi theo bit / α (RTN giả lập theo nhóm) → tìm vùng chuyển tiếp.
4. Chạy thử mọi đường ống của C1/C2/C3b: dùng lại hướng FP trên Q (b), học lại trên Q (b'), tách đoạn trước/sau layer (C2), sàn Pythia step0, dao động seed.

**Cách chạy mặc định:** không mở notebook này trên Kaggle. Sửa `configs/kaggle_run.yaml` → commit → Kaggle runner → *Save Version → Save & Run All*. Runner dùng papermill ghi đè ô `parameters` bên dưới.

**Chạy từng ô (ngoại lệ, xem QT-11):** import notebook này vào Kaggle, bật GPU + Internet, sửa ô `parameters`, chạy lần lượt. Nhớ tắt session khi xong.

**Kiểm code không cần GPU:** đặt `TOY_TEST = True` (model và dữ liệu giả, chạy CPU trong vài giây).

Quy ước: mốc FP = fp32; lượng tử hoá giả lập (quantize → dequantize) trên 4 ma trận Linear mỗi khối GPT-NeoX; embedding, lm head, LayerNorm giữ nguyên; W(α) = W + α·(Q(W) − W).

**Phải đối chiếu với repo gốc CausalGym trước khi tin số odds:** có thêm BOS không; cách tách token theo vùng; hàm loss khi học DAS (ở đây: cross-entropy về nhãn source); công thức odds (ở đây: [logit(y_src) − logit(y_base)] sau can thiệp trừ trước can thiệp).

In [1]:
# parameters — papermill ghi đè các giá trị này từ configs/kaggle_run.yaml
MODEL = "EleutherAI/pythia-410m"
REVISION = "main"
TASK = "agr_sv_num_subj-relc"
EVAL_SPLIT = "validation"
MAX_TRAIN = 400          # số câu train (bài gốc: khoảng 400)
MAX_EVAL = 200
LAYERS = "auto"          # "auto": 5 layer cách đều; hoặc list, ví dụ [6, 12]
REGIONS = "auto"         # "auto": vùng khác nhau giữa base/src và vùng cuối; hoặc list
MAIN_LAYER = -1          # -1: chọn ô FP có odds cao nhất (trừ layer cuối)
MAIN_REGION = -1
BITS = [8, 6, 4, 3]      # RTN đầy đủ, alpha 1
GROUP = 128              # group size; số không dương: lượng tử hoá theo hàng
ALPHA_BITS = 4
ALPHAS = [0.25, 0.5, 0.75]
RELEARN_BITS = [8, 4, 3] # học lại DAS trên Q ở ô chính
SPLIT_BITS = 4           # bit cho thí nghiệm tách đoạn C2
SEEDS = 3                # số seed ở ô chính (tiền thân null C1)
BS = 4
EVAL_BS = 16
EPOCHS = 1
LR = 5e-3
DTYPE = "float32"
SKIP_STEP0 = False
OUT_DIR = "outputs/g0_local"
REPO_DIR = "."
TOY_TEST = False         # True: model và dữ liệu giả trên CPU, chỉ để kiểm code

In [2]:
# Parameters
MODEL = "EleutherAI/pythia-1b"
TASK = "agr_sv_num_subj-relc"
LAYERS = [8]
BITS = [4]
ALPHAS = []
RELEARN_BITS = []
SEEDS = 1
SKIP_STEP0 = True
OUT_DIR = "/kaggle/working/outputs/20261112_1441_g0_full/g0_timing_1b"
REPO_DIR = "/tmp/repo"


## 1. Môi trường

In [3]:
import json, math, os, random, time
from collections import Counter
from types import SimpleNamespace

import numpy as np
import torch
import torch.nn.functional as F
import transformers

DEV = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs(OUT_DIR, exist_ok=True)
print("torch", torch.__version__, "| transformers", transformers.__version__, "| device", DEV,
      "|", torch.cuda.get_device_name(0) if DEV == "cuda" else "CPU")
if DEV == "cpu" and not TOY_TEST:
    print("CẢNH BÁO: không thấy GPU. Kiểm tra Settings → Accelerator trên Kaggle.")

torch 2.11.0+cu130 | transformers 5.20.2 | device cuda | Tesla T4


## 2. Hàm tiện ích

In [4]:
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

def sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

class Logger:
    # Ghi mỗi thí nghiệm thành một dòng JSONL ngay khi xong, để phiên bị ngắt vẫn giữ kết quả.
    def __init__(self, path):
        self.path = path
    def __call__(self, rec):
        rec = dict(rec, time=time.strftime("%Y-%m-%d %H:%M:%S"))
        print(json.dumps({k: v for k, v in rec.items() if not isinstance(v, list)}, ensure_ascii=False), flush=True)
        with open(self.path, "a", encoding="utf-8") as f:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

def spearman(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    if len(a) < 3:
        return float("nan")
    ra, rb = np.argsort(np.argsort(a)), np.argsort(np.argsort(b))
    return float(np.corrcoef(ra, rb)[0, 1])

log = Logger(os.path.join(OUT_DIR, "results.jsonl"))

## 3. Dữ liệu CausalGym

Mỗi câu là danh sách vùng (span). Tách token từng vùng rồi nối, ghi lại token cuối của mỗi vùng; can thiệp tại token cuối của vùng r.

In [5]:
def load_rows(task, split):
    from datasets import load_dataset
    ds = load_dataset("aryaman/causalgym", split=split)
    rows = [r for r in ds if r["task"] == task]
    if not rows:
        raise ValueError(f"Không có task '{task}'. Các task có: {sorted(set(ds['task']))}")
    return rows

def encode_spans(tok, spans):
    ids, ends = [], []
    for s in spans:
        ids += tok(s, add_special_tokens=False)["input_ids"]
        ends.append(len(ids) - 1)
    return ids, ends

def build_examples(tok, rows):
    exs, multi_tok = [], 0
    for r in rows:
        b_ids, b_end = encode_spans(tok, r["base"])
        s_ids, s_end = encode_spans(tok, r["src"])
        yb_t = tok(r["base_label"], add_special_tokens=False)["input_ids"]
        ys_t = tok(r["src_label"], add_special_tokens=False)["input_ids"]
        multi_tok += int(len(yb_t) > 1) + int(len(ys_t) > 1)
        if len(b_end) != len(s_end) or yb_t[0] == ys_t[0]:
            continue
        diff = [i for i, (a, b) in enumerate(zip(r["base"], r["src"])) if a != b]
        exs.append(dict(b_ids=b_ids, s_ids=s_ids, b_end=b_end, s_end=s_end, yb=yb_t[0], ys=ys_t[0], diff=diff))
    return exs, multi_tok

def find_regions(exs):
    n_regions = Counter(len(e["b_end"]) for e in exs).most_common(1)[0][0]
    diffs = Counter(e["diff"][0] for e in exs if e["diff"])
    return n_regions, (diffs.most_common(1)[0][0] if diffs else 0)

def _pad(seqs, pad_id, dev):
    L = max(len(s) for s in seqs)
    ids = torch.full((len(seqs), L), pad_id, dtype=torch.long)
    mask = torch.zeros((len(seqs), L), dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, :len(s)] = torch.tensor(s)
        mask[i, :len(s)] = 1
    last = torch.tensor([len(s) - 1 for s in seqs])
    return ids.to(dev), mask.to(dev), last.to(dev)

def make_batch(exs, region, pad_id, dev):
    b_ids, b_mask, b_last = _pad([e["b_ids"] for e in exs], pad_id, dev)
    s_ids, s_mask, s_last = _pad([e["s_ids"] for e in exs], pad_id, dev)
    t = lambda xs: torch.tensor(xs, device=dev)
    return dict(b_ids=b_ids, b_mask=b_mask, b_last=b_last, s_ids=s_ids, s_mask=s_mask, s_last=s_last,
                b_pos=t([e["b_end"][region] for e in exs]), s_pos=t([e["s_end"][region] for e in exs]),
                yb=t([e["yb"] for e in exs]), ys=t([e["ys"] for e in exs]))

## 4. Can thiệp 1D DII và DAS

Hook vào output của `gpt_neox.layers[l]` (residual stream sau layer l). Can thiệp chỉ thay toạ độ dọc hướng đơn vị v: h ← h + ((h_src − h) · v) v.

In [6]:
class Site:
    def __init__(self, model, layer):
        self.mode, self.pos, self.src, self.v, self.captured = None, None, None, None, None
        self.handle = model.gpt_neox.layers[layer].register_forward_hook(self._hook)
    def remove(self):
        self.handle.remove()
    def _hook(self, module, inputs, out):
        if self.mode is None:
            return None
        hs = out[0] if isinstance(out, tuple) else out
        idx = torch.arange(hs.shape[0], device=hs.device)
        if self.mode == "capture":
            self.captured = hs[idx, self.pos].detach()
            return None
        v = self.v.to(hs.dtype)
        hb = hs[idx, self.pos]
        coef = (self.src.to(hs.dtype) - hb) @ v
        new = hs.clone()
        new[idx, self.pos] = hb + coef[:, None] * v[None, :]
        return (new,) + tuple(out[1:]) if isinstance(out, tuple) else new

def last_logits(model, ids, mask, last):
    out = model(input_ids=ids, attention_mask=mask)
    return out.logits[torch.arange(ids.shape[0], device=ids.device), last].float()

def run_intervened(model, site, batch, v):
    with torch.no_grad():
        site.mode, site.pos = "capture", batch["s_pos"]
        model(input_ids=batch["s_ids"], attention_mask=batch["s_mask"])
    site.mode, site.pos, site.src, site.v = "intervene", batch["b_pos"], site.captured, v
    try:
        return last_logits(model, batch["b_ids"], batch["b_mask"], batch["b_last"])
    finally:
        site.mode = None

def train_das(model, layer, region, exs, seed, args, dev):
    # Học một hướng đơn vị v. Trả về (v, giây, loss cuối).
    set_seed(seed)
    d = model.config.hidden_size
    w = torch.nn.Parameter(torch.randn(d, device=dev) / math.sqrt(d))
    opt = torch.optim.Adam([w], lr=args.lr)
    order = list(range(len(exs))); random.Random(seed).shuffle(order)
    batches = [order[i:i + args.bs] for i in range(0, len(order), args.bs)] * args.epochs
    total, warm = len(batches), max(1, int(0.1 * len(batches)))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min((s + 1) / warm, max(0.0, (total - s) / max(1, total - warm))))
    site, loss = Site(model, layer), torch.tensor(float("nan"))
    sync(); t0 = time.time()
    try:
        for bi in batches:
            batch = make_batch([exs[i] for i in bi], region, args.pad_id, dev)
            logits = run_intervened(model, site, batch, w / w.norm())
            loss = F.cross_entropy(logits, batch["ys"])
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    finally:
        site.remove()
    sync()
    return (w / w.norm()).detach(), time.time() - t0, float(loss.item())

@torch.no_grad()
def eval_das(model, layer, region, exs, v, args, dev):
    site, odds, hits = Site(model, layer), [], []
    try:
        for i in range(0, len(exs), args.eval_bs):
            batch = make_batch(exs[i:i + args.eval_bs], region, args.pad_id, dev)
            ar = torch.arange(batch["yb"].shape[0], device=dev)
            base = last_logits(model, batch["b_ids"], batch["b_mask"], batch["b_last"])
            inter = run_intervened(model, site, batch, v)
            lo_b = base[ar, batch["ys"]] - base[ar, batch["yb"]]
            lo_i = inter[ar, batch["ys"]] - inter[ar, batch["yb"]]
            odds += (lo_i - lo_b).tolist(); hits += (lo_i > 0).float().tolist()
    finally:
        site.remove()
    return float(np.mean(odds)), float(np.mean(hits))

@torch.no_grad()
def eval_behavior(model, exs, args, dev):
    # Logit diff (nhãn đúng − nhãn sai) trên cả câu base và câu source.
    ld = []
    for i in range(0, len(exs), args.eval_bs):
        b = make_batch(exs[i:i + args.eval_bs], 0, args.pad_id, dev)
        ar = torch.arange(b["yb"].shape[0], device=dev)
        lb = last_logits(model, b["b_ids"], b["b_mask"], b["b_last"])
        ls = last_logits(model, b["s_ids"], b["s_mask"], b["s_last"])
        ld += (lb[ar, b["yb"]] - lb[ar, b["ys"]]).tolist()
        ld += (ls[ar, b["ys"]] - ls[ar, b["yb"]]).tolist()
    ld = np.asarray(ld)
    return dict(acc=float((ld > 0).mean()), logit_diff=float(ld.mean()), ld_sd=float(ld.std()),
                n=int(len(ld))), np.round(ld, 4).tolist()

## 5. Lượng tử hoá giả lập và nạp model

In [7]:
LIN_KEYS = ("attention.query_key_value", "attention.dense", "mlp.dense_h_to_4h", "mlp.dense_4h_to_h")

def rtn_group(W, bits, group):
    # RTN bất đối xứng theo nhóm dọc chiều input.
    out_f, in_f = W.shape
    g = in_f if (group <= 0 or in_f % group) else group
    Wg = W.reshape(out_f, in_f // g, g)
    lo, hi = Wg.amin(-1, keepdim=True), Wg.amax(-1, keepdim=True)
    qmax = 2 ** bits - 1
    scale = (hi - lo).clamp(min=1e-8) / qmax
    zero = torch.round(-lo / scale)
    q = torch.clamp(torch.round(Wg / scale) + zero, 0, qmax)
    return ((q - zero) * scale).reshape(out_f, in_f)

class Quantizer:
    # Giữ bản FP của các Linear trên CPU; set() luôn khôi phục FP rồi mới lượng tử hoá.
    def __init__(self, model):
        self.model, self.orig = model, {}
        for i, layer in enumerate(model.gpt_neox.layers):
            for k in LIN_KEYS:
                self.orig[(i, k)] = layer.get_submodule(k).weight.detach().to("cpu", copy=True)
    @torch.no_grad()
    def set(self, bits=None, group=128, alpha=1.0, layers=None):
        num = den = 0.0
        for (i, k), W0 in self.orig.items():
            p = self.model.gpt_neox.layers[i].get_submodule(k).weight
            W = W0.to(p.device, torch.float32)
            if bits is not None and (layers is None or i in layers):
                Wn = W + alpha * (rtn_group(W, bits, group) - W)
                num += float((Wn - W).pow(2).sum()); den += float(W.pow(2).sum())
                W = Wn
            p.copy_(W.to(p.dtype))
        return math.sqrt(num / den) if den > 0 else 0.0   # sai số tương đối của trọng số

def load_model(name, revision, dtype, dev):
    from transformers import AutoModelForCausalLM
    m = AutoModelForCausalLM.from_pretrained(name, revision=revision)
    m = m.to(dev, dtype)          # ép dtype tường minh
    m.eval()
    for p in m.parameters():
        p.requires_grad_(False)
    assert hasattr(m, "gpt_neox"), "Notebook chỉ hỗ trợ Pythia / GPT-NeoX"
    return m

## 6. Nạp dữ liệu và model

Nếu `TOY_TEST = True`, ô này thay dữ liệu, tokenizer và model bằng bản giả (không tải gì).

In [8]:
if TOY_TEST:
    from transformers import GPTNeoXConfig, GPTNeoXForCausalLM
    class ToyTok:
        pad_token_id, eos_token_id, vocab = None, 0, {}
        def __call__(self, s, add_special_tokens=False):
            ids = []
            for w in s.split():
                self.vocab.setdefault(w, 2 + len(self.vocab) % 100)
                ids.append(self.vocab[w])
            return {"input_ids": ids or [1]}
    def load_rows(task, split):
        rng, out = random.Random(split), []
        for _ in range(60):
            sg = rng.random() < 0.5
            b = [" The", " man" if sg else " men", " near the car", " that sleeps"]
            s = [" The", " men" if sg else " man", " near the car", " that sleeps"]
            out.append(dict(base=b, src=s, base_label=" is" if sg else " are",
                            src_label=" are" if sg else " is", task=task))
        return out
    def load_model(name, revision, dtype, dev):
        torch.manual_seed(1 if revision == "step0" else 0)
        cfg = GPTNeoXConfig(vocab_size=120, hidden_size=64, num_hidden_layers=6, num_attention_heads=4,
                            intermediate_size=256, max_position_embeddings=64)
        m = GPTNeoXForCausalLM(cfg).to(dev, dtype).eval()
        for p in m.parameters():
            p.requires_grad_(False)
        return m
    tok = ToyTok()
else:
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(MODEL, revision=REVISION)

args = SimpleNamespace(lr=LR, bs=BS, eval_bs=EVAL_BS, epochs=EPOCHS,
                       pad_id=tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id)
dtype = getattr(torch, DTYPE)

tr, mt1 = build_examples(tok, load_rows(TASK, "train"))
ev, mt2 = build_examples(tok, load_rows(TASK, EVAL_SPLIT))
n_reg, diff_reg = find_regions(tr)
tr = [e for e in tr if len(e["b_end"]) == n_reg][:MAX_TRAIN]
ev = [e for e in ev if len(e["b_end"]) == n_reg][:MAX_EVAL]

model = load_model(MODEL, REVISION, dtype, DEV)
L = model.config.num_hidden_layers
layers = sorted({int(round(x)) for x in np.linspace(0, L - 1, 5)}) if LAYERS == "auto" else list(LAYERS)
regions = sorted({diff_reg, n_reg - 1}) if REGIONS == "auto" else list(REGIONS)
sites = [(l, r) for l in layers for r in regions]

log(dict(exp="meta", model=MODEL, revision=REVISION, task=TASK, toy=TOY_TEST,
         device=torch.cuda.get_device_name(0) if DEV == "cuda" else "cpu",
         torch=torch.__version__, transformers=transformers.__version__, dtype=DTYPE,
         n_layers=L, n_train=len(tr), n_eval=len(ev), n_regions=n_reg, diff_region=diff_reg,
         multi_token_labels=mt1 + mt2, layers=layers, regions=regions))

config.json:   0%|          | 0.00/569 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

{"exp": "meta", "model": "EleutherAI/pythia-1b", "revision": "main", "task": "agr_sv_num_subj-relc", "toy": false, "device": "Tesla T4", "torch": "2.11.0+cu130", "transformers": "5.20.2", "dtype": "float32", "n_layers": 16, "n_train": 400, "n_eval": 100, "n_regions": 6, "diff_region": 1, "multi_token_labels": 0, "time": "2026-11-12 14:46:23"}


## 7. A — Hành vi FP

In [9]:
beh, ld = eval_behavior(model, ev, args, DEV)
log(dict(exp="behavior", bits=None, alpha=0.0, **beh, ld=ld))

{"exp": "behavior", "bits": null, "alpha": 0.0, "acc": 0.99, "logit_diff": 3.9565362310409546, "ld_sd": 1.487501166635968, "n": 200, "time": "2026-11-12 14:46:25"}


## 8. B — DAS trên FP ở mọi ô (odds, IIA, thời gian một lần học)

In [10]:
fp_dirs, fp_odds, t_train, t_eval = {}, {}, [], []
for (l, r) in sites:
    v, tt, loss = train_das(model, l, r, tr, 0, args, DEV)
    sync(); t0 = time.time()
    odds, iia = eval_das(model, l, r, ev, v, args, DEV)
    sync(); te = time.time() - t0
    fp_dirs[(l, r)], fp_odds[(l, r)] = v, odds
    t_train.append(tt); t_eval.append(te)
    log(dict(exp="das_fp", layer=l, region=r, seed=0, odds=odds, iia=iia, final_loss=loss,
             t_train_s=round(tt, 2), t_eval_s=round(te, 2)))

{"exp": "das_fp", "layer": 8, "region": 1, "seed": 0, "odds": 5.361465363502503, "iia": 0.75, "final_loss": 4.683976650238037, "t_train_s": 7.33, "t_eval_s": 1.82, "time": "2026-11-12 14:46:34"}


{"exp": "das_fp", "layer": 8, "region": 5, "seed": 0, "odds": 1.3115930986404418, "iia": 0.09, "final_loss": 7.860745429992676, "t_train_s": 7.34, "t_eval_s": 1.92, "time": "2026-11-12 14:46:43"}


## 9. Chọn ô chính

In [11]:
if MAIN_LAYER >= 0:
    main = (MAIN_LAYER, MAIN_REGION if MAIN_REGION >= 0 else regions[-1])
    if main not in fp_dirs:
        v, _, _ = train_das(model, main[0], main[1], tr, 0, args, DEV)
        fp_dirs[main] = v
        fp_odds[main] = eval_das(model, main[0], main[1], ev, v, args, DEV)[0]
else:
    cand = [s for s in sites if s[0] != L - 1] or sites
    main = max(cand, key=lambda s: fp_odds[s])
ml, mr = main
log(dict(exp="main_site", layer=ml, region=mr, odds=fp_odds[main]))

{"exp": "main_site", "layer": 8, "region": 1, "odds": 5.361465363502503, "time": "2026-11-12 14:46:43"}


## 10. C — Dao động theo seed ở ô chính (tiền thân null C1)

In [12]:
seed_dirs, seed_odds = [fp_dirs[main]], [fp_odds[main]]
for s in range(1, SEEDS):
    v, tt, _ = train_das(model, ml, mr, tr, s, args, DEV)
    odds, iia = eval_das(model, ml, mr, ev, v, args, DEV)
    seed_dirs.append(v); seed_odds.append(odds); t_train.append(tt)
    log(dict(exp="das_fp_seed", layer=ml, region=mr, seed=s, odds=odds, iia=iia, t_train_s=round(tt, 2)))
cos = [abs(float(seed_dirs[i] @ seed_dirs[j])) for i in range(len(seed_dirs)) for j in range(i + 1, len(seed_dirs))]
log(dict(exp="seed_spread", layer=ml, region=mr, odds_min=min(seed_odds), odds_max=max(seed_odds),
         abs_cos_min=min(cos) if cos else None, abs_cos_mean=float(np.mean(cos)) if cos else None))

{"exp": "seed_spread", "layer": 8, "region": 1, "odds_min": 5.361465363502503, "odds_max": 5.361465363502503, "abs_cos_min": null, "abs_cos_mean": null, "time": "2026-11-12 14:46:43"}


## 11. D + E — Lượng tử hoá: hành vi, dùng lại hướng FP (b), bản đồ (d), học lại (b')

In [13]:
quant = Quantizer(model)
configs = [(b, 1.0) for b in BITS] + [(ALPHA_BITS, a) for a in ALPHAS]
site_list = list(fp_odds.keys())
for bits, alpha in configs:
    rel_err = quant.set(bits, GROUP, alpha)
    beh, ld = eval_behavior(model, ev, args, DEV)
    log(dict(exp="behavior", bits=bits, group=GROUP, alpha=alpha, w_rel_err=rel_err, **beh, ld=ld))
    reuse = {s: eval_das(model, s[0], s[1], ev, fp_dirs[s], args, DEV)[0] for s in site_list}
    log(dict(exp="reuse_map", bits=bits, group=GROUP, alpha=alpha,
             spearman_vs_fp=spearman([fp_odds[s] for s in site_list], [reuse[s] for s in site_list]),
             main_odds_fp=fp_odds[main], main_odds_reuse=reuse[main],
             transfer_T=reuse[main] / fp_odds[main] if fp_odds[main] else None,
             map=[[s[0], s[1], fp_odds[s], reuse[s]] for s in site_list]))
    if alpha == 1.0 and bits in RELEARN_BITS:
        v, tt, _ = train_das(model, ml, mr, tr, 0, args, DEV)
        odds, iia = eval_das(model, ml, mr, ev, v, args, DEV)
        t_train.append(tt)
        log(dict(exp="relearn_q", bits=bits, group=GROUP, layer=ml, region=mr, odds=odds, iia=iia,
                 abs_cos_vs_fp=abs(float(v @ fp_dirs[main])), t_train_s=round(tt, 2)))

{"exp": "behavior", "bits": 4, "group": 128, "alpha": 1.0, "w_rel_err": 0.1022872381167501, "acc": 0.99, "logit_diff": 4.113624753952027, "ld_sd": 1.4736202791763597, "n": 200, "time": "2026-11-12 14:46:49"}


{"exp": "reuse_map", "bits": 4, "group": 128, "alpha": 1.0, "spearman_vs_fp": NaN, "main_odds_fp": 5.361465363502503, "main_odds_reuse": 5.379993758201599, "transfer_T": 1.0034558452667113, "time": "2026-11-12 14:46:53"}


## 12. F — C2: tách đoạn (chỉ lượng tử hoá phía mã hoá layer ≤ l, hoặc phía đọc ra layer > l)

In [14]:
for name, lay in [("pre", set(range(0, ml + 1))), ("post", set(range(ml + 1, L))), ("all", None)]:
    rel_err = quant.set(SPLIT_BITS, GROUP, 1.0, layers=lay)
    beh, _ = eval_behavior(model, ev, args, DEV)
    odds, iia = eval_das(model, ml, mr, ev, fp_dirs[main], args, DEV)
    log(dict(exp="split_q", part=name, bits=SPLIT_BITS, layer=ml, region=mr, w_rel_err=rel_err,
             acc=beh["acc"], logit_diff=beh["logit_diff"], odds_reuse=odds, iia_reuse=iia, odds_fp=fp_odds[main]))
quant.set(None)   # khôi phục FP

{"exp": "split_q", "part": "pre", "bits": 4, "layer": 8, "region": 1, "w_rel_err": 0.10195834899565558, "acc": 0.99, "logit_diff": 4.102293796539307, "odds_reuse": 5.543913869857788, "iia_reuse": 0.75, "odds_fp": 5.361465363502503, "time": "2026-11-12 14:46:57"}


{"exp": "split_q", "part": "post", "bits": 4, "layer": 8, "region": 1, "w_rel_err": 0.10262219492143637, "acc": 0.99, "logit_diff": 3.9852591609954833, "odds_reuse": 5.229997062683106, "iia_reuse": 0.73, "odds_fp": 5.361465363502503, "time": "2026-11-12 14:47:02"}


{"exp": "split_q", "part": "all", "bits": 4, "layer": 8, "region": 1, "w_rel_err": 0.1022872381167501, "acc": 0.99, "logit_diff": 4.113624753952027, "odds_reuse": 5.379993758201599, "iia_reuse": 0.76, "odds_fp": 5.361465363502503, "time": "2026-11-12 14:47:07"}


0.0

## 13. G — Sàn: DAS trên Pythia step0 (model chưa học gì) ở ô chính

In [15]:
if not SKIP_STEP0:
    del quant, model
    if DEV == "cuda":
        torch.cuda.empty_cache()
    m0 = load_model(MODEL, "step0", dtype, DEV)
    v, tt, _ = train_das(m0, ml, mr, tr, 0, args, DEV)
    odds, iia = eval_das(m0, ml, mr, ev, v, args, DEV)
    log(dict(exp="floor_step0", layer=ml, region=mr, odds=odds, iia=iia, t_train_s=round(tt, 2),
             ratio_to_fp=odds / fp_odds[main] if fp_odds[main] else None))
    del m0

## 14. Thời gian và tóm tắt

In [16]:
t_run = float(np.median(t_train)) + float(np.median(t_eval))
log(dict(exp="timing", t_train_median_s=round(float(np.median(t_train)), 2),
         t_eval_median_s=round(float(np.median(t_eval)), 2), t_per_run_s=round(t_run, 2),
         gpu_hours_3000_runs=round(3000 * t_run / 3600, 1), gpu_hours_10000_runs=round(10000 * t_run / 3600, 1)))

import pandas as pd
recs = [json.loads(x) for x in open(os.path.join(OUT_DIR, "results.jsonl"), encoding="utf-8")]
beh_df = pd.DataFrame([r for r in recs if r["exp"] == "behavior"])[["bits", "alpha", "acc", "logit_diff"]]
reuse_df = pd.DataFrame([r for r in recs if r["exp"] == "reuse_map"])[
    ["bits", "alpha", "spearman_vs_fp", "main_odds_fp", "main_odds_reuse", "transfer_T"]]
print("\nHành vi theo mức nén:\n", beh_df.to_string(index=False))
print("\nDùng lại hướng FP:\n", reuse_df.to_string(index=False))
beh_df.to_csv(os.path.join(OUT_DIR, "summary_behavior.csv"), index=False)
reuse_df.to_csv(os.path.join(OUT_DIR, "summary_reuse.csv"), index=False)
print("\nXong. Kết quả trong", OUT_DIR)

{"exp": "timing", "t_train_median_s": 7.34, "t_eval_median_s": 1.87, "t_per_run_s": 9.21, "gpu_hours_3000_runs": 7.7, "gpu_hours_10000_runs": 25.6, "time": "2026-11-12 14:47:07"}



Hành vi theo mức nén:
  bits  alpha  acc  logit_diff
  NaN    0.0 0.99    3.956536
  4.0    1.0 0.99    4.113625

Dùng lại hướng FP:
  bits  alpha  spearman_vs_fp  main_odds_fp  main_odds_reuse  transfer_T
    4    1.0             NaN      5.361465         5.379994    1.003456

Xong. Kết quả trong /kaggle/working/outputs/20261112_1441_g0_full/g0_timing_1b
